# 클러스터링 기반 서브그룹 피처

IVF/DI 서브모델 분리(모델 자체를 쪼개는 방식)는 DI 표본 부족으로 실패했습니다.
이번에는 모델을 쪼개지 않고, 대신 **비지도 클러스터링으로 환자군을 부드럽게
나눈 뒤 그 클러스터 ID를 피처로 추가**하는 방식을 시도합니다. 모델은 여전히
하나(LightGBM)이고, 클러스터 정보는 추가 신호로만 작동하므로 서브모델 분리보다
안전합니다.

핵심 가정: 나이, 배아 수, 시술 이력 등을 기준으로 비슷한 환자들을 그룹화하면,
그 그룹 정체성 자체가 (개별 피처들의 단순 조합보다) 더 압축된 신호를 줄 수
있다는 것입니다. 다만 이전 실험들에서 명시적 교차피처가 거의 항상 효과
없었던 것을 고려하면, 이 시도도 비슷한 결과(효과 없음)가 나올 가능성이
높습니다 — 그래도 확인할 가치가 있습니다.


## 1. 라이브러리 및 설정

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

## 2. 데이터 로드 및 전처리 (main.py와 동일)

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


def encode_cat(df):
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df


df_base = encode_cat(fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS)))
X_base = df_base.drop(columns=[TARGET_COL])
y = df_base[TARGET_COL]
print(f"전처리 완료: {X_base.shape}")

## 3. 클러스터링용 피처 선정 및 클러스터 생성

EDA에서 확인된 핵심 신호 피처(나이, 이식배아수, 시술유형, 이전 시술 이력 등)
기준으로 클러스터링합니다. 클러스터 개수(k)는 5, 8, 12로 각각 시도합니다.


In [ ]:
cluster_feature_names = [
    "시술 당시 나이", "이식된 배아 수", "총 생성 배아 수", "is_DI",
    "총 시술 횟수", "배아 이식 경과일", "동결 배아 사용 여부",
]
cluster_features = X_base[cluster_feature_names].copy()

scaler = StandardScaler()
cluster_features_scaled = scaler.fit_transform(cluster_features)

print(f"클러스터링용 피처: {cluster_feature_names}")

## 4. 클러스터 개수(k)별 효과 검증

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)

results = {}

# 대조군 (클러스터 피처 없음)
scores = []
for tr_idx, val_idx in skf.split(X_base, y):
    model = LGBMClassifier(random_state=1004, verbose=-1)
    model.fit(X_base.iloc[tr_idx], y.iloc[tr_idx])
    proba = model.predict_proba(X_base.iloc[val_idx])[:, 1]
    scores.append(roc_auc_score(y.iloc[val_idx], proba))
results["대조군"] = np.mean(scores)
print(f"대조군 (클러스터 피처 없음): {results['대조군']:.5f}")

for k in [5, 8, 12]:
    kmeans = KMeans(n_clusters=k, random_state=1004, n_init=10)
    cluster_labels = kmeans.fit_predict(cluster_features_scaled)

    X_with_cluster = X_base.copy()
    X_with_cluster["cluster_id"] = cluster_labels

    scores = []
    for tr_idx, val_idx in skf.split(X_with_cluster, y):
        model = LGBMClassifier(random_state=1004, verbose=-1)
        model.fit(X_with_cluster.iloc[tr_idx], y.iloc[tr_idx])
        proba = model.predict_proba(X_with_cluster.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], proba))
    results[f"k={k}"] = np.mean(scores)
    print(f"k={k} 클러스터 피처 추가: {results[f'k={k}']:.5f}  (차이: {results[f'k={k}'] - results['대조군']:+.5f})")

## 5. 결론

개선이 노이즈 수준(std 0.002 미만)이면, 이번에도 LightGBM이 이미 원본
피처들로 비슷한 그룹 구분을 스스로 학습하고 있었다는 뜻입니다. 의미 있는
개선이 확인되면 k를 더 세밀하게(예: 3~20 범위) 탐색해보거나, 클러스터링
피처 구성을 바꿔서 추가로 검증해볼 수 있습니다.
